# WaferGuard ML — Financial Impact Analysis
### Phase 2: Anomaly Detection → Batch Frequency → Financial Recommendations

This notebook is the direct continuation of `model_testing.ipynb`.  
It takes the trained models and produces a **ranked financial impact report** for a production batch.

**Pipeline:**
```
    → Run model on full batch
    → Count detections per pattern  ← frequency pre-step
    → Join to financial mapping table
    → Compute weighted daily loss, break-even, EVoA, priority score
    → Ranked repair recommendations
```
---

In [1]:
import json
import sys
from pathlib import Path

import numpy as np

In [2]:
# ── Configuration: Batch settings and financial assumptions ──────────────────
BATCH_ID = "LOT_001"  # Production lot identifier
SELECTED_MODEL = "cnn"  # Which model to use: "cnn" or "tl"
CONFIDENCE_THRESHOLD = 0.70  # Flag predictions below this for manual review
WPH = 100  # Wafers per hour through affected tool
VALUE_PER_WAFER = 5_000  # USD per wafer
REPAIR_HOURS = 8  # Estimated downtime hours per repair event
PLANNING_HORIZON = 30  # Days used for EVoA calculation

print("✓ Configuration set")
print(f"  Batch ID         : {BATCH_ID}")
print(f"  Selected model   : {SELECTED_MODEL.upper()}")
print(f"  WPH              : {WPH}")
print(f"  Value / wafer    : ${VALUE_PER_WAFER:,}")
print(f"  Conf. threshold  : {CONFIDENCE_THRESHOLD:.0%}")
print(f"  Planning horizon : {PLANNING_HORIZON} days")

✓ Configuration set
  Batch ID         : LOT_001
  Selected model   : CNN
  WPH              : 100
  Value / wafer    : $5,000
  Conf. threshold  : 70%
  Planning horizon : 30 days


## Production Pipeline: Run Refactored Library
Execute the refactored financial analysis pipeline using the `financial_impact.financial` library.  
Set configuration above, then run this cell to generate reports.

In [6]:
# Run financial analysis pipeline using trained models + dataset
from financial_impact import financial
from financial_impact import inference as wg_inf


def _find_repo_root() -> Path:
    """Walk up from cwd until pyproject.toml is found."""
    p = Path.cwd()
    for _ in range(6):
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not locate repo root (pyproject.toml not found)")


repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)


models_dir = repo_root / "wafer_images" / "phase2_models"

label_mapping = {
    "00000000": "Normal",
    "10000000": "Center",
    "01000000": "Donut",
    "00100000": "Edge_Loc",
    "00010000": "Edge_Ring",
    "00001000": "Loc",
    "00000100": "Near_Full",
    "00000010": "Scratch",
    "00000001": "Random",
    "10100000": "Center+Edge_Loc",
    "10010000": "Center+Edge_Ring",
    "10001000": "Center+Loc",
    "10000010": "Center+Scratch",
    "01100000": "Donut+Edge_Loc",
    "01010000": "Donut+Edge_Ring",
    "01001000": "Donut+Loc",
    "01000010": "Donut+Scratch",
    "00101000": "Edge_Loc+Loc",
    "00100010": "Edge_Loc+Scratch",
    "00011000": "Edge_Ring+Loc",
    "00010010": "Edge_Ring+Scratch",
    "00001010": "Loc+Scratch",
    "10101000": "Center+Edge_Loc+Loc",
    "10100010": "Center+Edge_Loc+Scratch",
    "10011000": "Center+Edge_Ring+Loc",
    "10010010": "Center+Edge_Ring+Scratch",
    "10001010": "Center+Loc+Scratch",
    "01101000": "Donut+Edge_Loc+Loc",
    "01100010": "Donut+Edge_Loc+Scratch",
    "01011000": "Donut+Edge_Ring+Loc",
    "01010010": "Donut+Edge_Ring+Scratch",
    "01001010": "Donut+Loc+Scratch",
    "00101010": "Edge_Loc+Loc+Scratch",
    "00011010": "Edge_Ring+Loc+Scratch",
    "10101010": "Center+Edge_Loc+Loc+Scratch",
    "10011010": "Center+Edge_Ring+Loc+Scratch",
    "01101010": "Donut+Edge_Loc+Loc+Scratch",
    "01011010": "Donut+Edge_Ring+Loc+Scratch",
}

unique_patterns = sorted(label_mapping.keys())
pattern_to_id = {pattern: idx for idx, pattern in enumerate(unique_patterns)}
id_to_pattern = {idx: label_mapping[pattern] for pattern, idx in pattern_to_id.items()}
id_to_binary = {v: k for k, v in pattern_to_id.items()}

# Load dataset
data_path = repo_root / "data" / "mixedtype-wafer-defect-datasets" / "Wafer_Map_Datasets.npz"
print("Loading dataset from", data_path)
data = np.load(data_path)
images = data["arr_0"]
labels = data["arr_1"]
images[images == 3] = 0
X_all = np.eye(3, dtype=np.float32)[images.astype(int)]
label_str_arr = np.array(["".join(map(str, map(int, row))) for row in labels])
X_batch = X_all
print("Loaded dataset shape:", X_batch.shape)

# Run inference with the selected model
SELECTED_MODEL = globals().get("SELECTED_MODEL", "cnn")
model_filename = "model_p2_cnn.keras" if SELECTED_MODEL == "cnn" else "model_p2_tl.keras"
model_path = str(models_dir / model_filename)
print(f"Running inference with: {SELECTED_MODEL.upper()} ({model_path})")
predictions = wg_inf.run_model(model_path, X_batch)

# Decode + build batch summary
class_ids, confidences = financial.get_predictions(predictions)
binary_labels, pattern_names = financial.decode_labels(class_ids, id_to_binary, id_to_pattern)
df_batch2 = financial.build_batch_df(binary_labels, confidences, label_mapping)

# Financial config
BATCH_ID = globals().get("BATCH_ID", "LOT_001")
WPH = globals().get("WPH", 100)
VALUE_PER_WAFER = globals().get("VALUE_PER_WAFER", 5000)
REPAIR_HOURS = globals().get("REPAIR_HOURS", 8)
PLANNING_HORIZON = globals().get("PLANNING_HORIZON", 30)
CONFIDENCE_THRESHOLD = globals().get("CONFIDENCE_THRESHOLD", 0.7)

config = {
    "WPH": WPH,
    "VALUE_PER_WAFER": VALUE_PER_WAFER,
    "REPAIR_HOURS": REPAIR_HOURS,
    "PLANNING_HORIZON": PLANNING_HORIZON,
    "BATCH_ID": BATCH_ID,
    "total_wafers": len(binary_labels),
    "low_conf_count": int((confidences < CONFIDENCE_THRESHOLD).sum()),
}
df_financial2, summary2 = financial.compute_financials(df_batch2, config)

# Preview and save
print("--- Financial preview (top 5) ---")
print(df_financial2.head(5).to_string(index=False))
print()
print("--- Summary payload (truncated) ---")
print(json.dumps(summary2, indent=2)[:1000])
outdir_demo = str(repo_root / f"reports_{BATCH_ID}" / "refactor_demo")
financial.save_reports(df_batch2, df_financial2, summary2, outdir_demo, BATCH_ID)
print("Saved refactor reports →", outdir_demo)

repo_root: /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml
Loading dataset from /Users/hernanmontoyag/Desktop/Capstone/waferguard-ml/data/mixedtype-wafer-defect-datasets/Wafer_Map_Datasets.npz
Loaded dataset shape: (38015, 52, 52, 3)
Running inference with: CNN (/Users/hernanmontoyag/Desktop/Capstone/waferguard-ml/wafer_images/phase2_models/model_p2_cnn.keras)
--- Financial preview (top 5) ---
binary_label                pattern_name  count  batch_pct  avg_confidence  triage_priority risk_level  yield_loss_pct  repair_cost  replacement_cost  downtime_per_hr  weighted_daily_loss  break_even_days   evoa_30d  priority_score                                                                                                                                                             process_step                                                                                                                                                                        repair_action
    01101010  Do

## Financial Impact by Base Anomaly (Decomposed View)

The table above treats all 38 patterns as independent. This table **decomposes** every detected pattern into its 8 base anomaly types and aggregates counts at the base level.

**Logic:** A wafer classified as *Donut + Edge Lock + Scratch* contributes **+1** to each of Donut, Edge Lock, and Scratch independently.

**Why this matters:** Fixing a root cause (e.g., the CMP machine) resolves the base anomaly across *every* combination pattern that contains it. This table reflects the true aggregate financial impact of addressing each base anomaly independently — giving a repair-action-oriented view.

In [4]:
# Decompose all detections to base anomaly level and recompute financial metrics
df_batch_base = financial.build_base_anomaly_df(binary_labels, confidences, label_mapping)
df_financial_base, summary_base = financial.compute_financials(df_batch_base, config)

print("--- Financial Impact by Base Anomaly (Decomposed View) ---")
print(df_financial_base.to_string(index=False))

# Save decomposed report
base_csv_path = Path(outdir_demo) / f"base_anomaly_report_{BATCH_ID}.csv"
df_financial_base.to_csv(base_csv_path, index=False)
print(f"\nSaved base anomaly report → {base_csv_path}")

--- Financial Impact by Base Anomaly (Decomposed View) ---
binary_label pattern_name  count  batch_pct  avg_confidence  triage_priority risk_level  yield_loss_pct  repair_cost  replacement_cost  downtime_per_hr  weighted_daily_loss  break_even_days    evoa_30d  priority_score                                                process_step                                                                                                      repair_action
    00000010      Scratch  16221      42.67          0.9672                2   Critical            27.5        77500         5250000.0           550000           1408110.00              0.1  37765800.0       704055.00                           Wafer Handling Path / CMP Contact Immediate: contain handling path; swap/clean robot end effectors; inspect FOUPs/cassettes; replace CMP consumables
    01000000        Donut  11913      31.34          0.9756                4       High            32.5       130000         5250000.0           550000    

## Action-Oriented Financial Analysis
Restructures the output from **pattern-rows** to **repair-action rows**.

Each row = one manufacturing action (e.g., "Replace CMP polishing pad").  
Shows the **aggregate financial impact** of taking that action across all 38 patterns it resolves — fully or partially.

- **Fully resolved**: the action addresses all root-cause components of the pattern.  
- **Partially resolved**: the action fixes some components; savings are proportional to the fraction resolved.

Use this table to answer: *"If I take action X, what is the total financial gain?"*

In [5]:
# Build action-oriented financial table
df_actions = financial.compute_action_table(df_financial2)

print(f"Total unique repair actions identified: {len(df_actions)}")
print()
print("--- Action-Oriented Financial Table (top 10 by daily loss savings) ---")
print(df_actions.head(10).to_string(index=False))

# Save action table alongside the pattern-oriented report
action_path = Path(outdir_demo) / f"action_report_{BATCH_ID}.csv"
df_actions.to_csv(action_path, index=False)
print(f"\nSaved action report → {action_path}")

Total unique repair actions identified: 23

--- Action-Oriented Financial Table (top 10 by daily loss savings) ---
                                repair_action                                        process_step risk_level  triage_priority  repair_cost  patterns_fully_resolved  patterns_partially_resolved  daily_loss_savings  break_even_days    evoa_30d
              Furnace zone temp recalibration                      Diffusion / Thermal Processing     Medium                5       110000                        1                           10          1170072.16              0.1 34992164.65
                thermocouple/lamp replacement                      Diffusion / Thermal Processing     Medium                5       110000                        1                           10          1170072.16              0.1 34992164.65
                requalification thermal cycle                      Diffusion / Thermal Processing     Medium                5       110000                     